In [1]:
import mne
import numpy as np

# ==========================================
# 1. WCZYTANIE DANYCH I STANDARYZACJA
# ==========================================
# Pobieramy przykładowe dane: Badany nr 1, sesje 4, 8, 12 (wyobraźnia ruchowa rąk i nóg)
subject = 1
runs = [4, 8, 12]
raw_fnames = mne.datasets.eegbci.load_data(subject, runs)

# Wczytanie plików EDF
raws = [mne.io.read_raw_edf(f, preload=True) for f in raw_fnames]
raw = mne.concatenate_raws(raws)

# Standaryzacja nazw kanałów do systemu 10-05 i przypisanie montażu
mne.datasets.eegbci.standardize(raw)
montage = mne.channels.make_standard_montage('standard_1005')
raw.set_montage(montage)

# ==========================================
# 2. RE-REFERENCJA (CAR - Common Average Reference)
# ==========================================
# Standard w literaturze w celu usunięcia globalnego szumu środowiskowego
raw.set_eeg_reference('average', projection=False)

# ==========================================
# 3. FILTRACJA SIECI (Line Noise) I BAND-PASS
# ==========================================
# Dane pochodzą z USA, gdzie w gniazdkach jest 60 Hz.
# Częstotliwość próbkowania to 160 Hz (Nyquist = 80 Hz), więc interesuje nas tylko 60 Hz.
raw.notch_filter(freqs=[60.0])

# Filtr pasmowo-przepustowy: 1 Hz (usuwa wolne dryfty) do 40 Hz (wycina wysokie częstotliwości).
# W analizie ruchowej (ERD/ERS) najważniejsze pasma mieszczą się między 8 a 30 Hz.
raw.filter(l_freq=1.0, h_freq=40.0, fir_design='firwin')

# ==========================================
# 4. USUWANIE ARTEFAKTÓW (ICA)
# ==========================================
# Używamy Independent Component Analysis (ICA) do usunięcia mrugnięć (EOG) i EKG.
# Dataset nie ma dedykowanego kanału EOG, ale kanał Fp1 leży tuż nad okiem.
ica = mne.preprocessing.ICA(n_components=15, random_state=42, max_iter='auto')
ica.fit(raw)

# Automatyczne wykrywanie komponentów odpowiadających za mrugnięcia za pomocą kanału Fp1
eog_indices, eog_scores = ica.find_bads_eog(raw, ch_name='Fp1')
ica.exclude = eog_indices # Oznaczamy znalezione komponenty do usunięcia
ica.apply(raw)            # Rekonstrukcja sygnału bez artefaktów ocznych

# ==========================================
# 5. EPOKOWANIE (Podział na próby eksperymentalne)
# ==========================================
# Ekstrakcja markerów z anotacji BCI2000 (T0 - odpoczynek, T1 - lewa pięść, T2 - prawa pięść)
events, event_id = mne.events_from_annotations(raw)

# Wybieramy tylko kanały EEG
picks = mne.pick_types(raw.info, meg=False, eeg=True, stim=False, eog=False)

# Tworzymy epoki: od 0.5s przed bodźcem do 4.0s po bodźcu. Baseline correction przed bodźcem.
epochs = mne.Epochs(raw, events, event_id, tmin=-0.5, tmax=4.0, proj=True,
                    picks=picks, baseline=(None, 0), preload=True)

# ==========================================
# 6. USUWANIE ARTEFAKTÓW RUCHOWYCH (AutoReject) - ZAKTUALIZOWANE
# ==========================================
from autoreject import AutoReject

print("Uruchamianie AutoReject (może zająć chwilę)...")
# Inicjalizacja algorytmu - automatycznie znajdzie optymalne progi
ar = AutoReject(random_state=42)

# Dopasowanie i czyszczenie epok (algorytm sam naprawi co się da, a resztę wyrzuci)
epochs, reject_log = ar.fit_transform(epochs, return_log=True)

print(f"Pozostało epok po AutoReject: {len(epochs)}")

# ==========================================
# 7. PODZIAŁ NA PASMA CZĘSTOTLIWOŚCI
# ==========================================
# Generujemy osobne zestawy danych dla poszczególnych pasm, co jest kluczowe w BCI
freq_bands = {
    'Theta': (4.0, 8.0),
    'Mu/Alpha': (8.0, 13.0),  # Najważniejsze dla kory ruchowej (sensorimotor rhythm)
    'Beta': (13.0, 30.0),     # Ważne przy planowaniu i po zakończeniu ruchu
    'Low-Gamma': (30.0, 40.0) # Ponieważ odcięliśmy sygnał na 40 Hz
}

epochs_by_band = {}
for band, (l_freq, h_freq) in freq_bands.items():
    # Kopiujemy epoki i filtrujemy je do konkretnego pasma
    epochs_by_band[band] = epochs.copy().filter(l_freq=l_freq, h_freq=h_freq)

print("Pre-processing zakończony sukcesem!")
print(f"Pozostało epok po odrzuceniu artefaktów: {len(epochs)}")

Extracting EDF parameters from /Users/julia/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /Users/julia/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /Users/julia/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Filtering raw data in 3 contiguous segments
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
-

  0%|          | Creating augmented epochs : 0/64 [00:00<?,       ?it/s]

  0%|          | Computing thresholds ... : 0/64 [00:00<?,       ?it/s]

  0%|          | Repairing epochs : 0/87 [00:00<?,       ?it/s]

  0%|          | n_interp : 0/3 [00:00<?,       ?it/s]

...

  0%|          | Repairing epochs : 0/87 [00:00<?,       ?it/s]

  0%|          | Fold : 0/10 [00:00<?,       ?it/s]





Estimated consensus=0.30 and n_interpolate=1


  0%|          | Repairing epochs : 0/87 [00:00<?,       ?it/s]

No bad epochs were found for your data. Returning a copy of the data you wanted to clean. Interpolation may have been done.
Pozostało epok po AutoReject: 87
Setting up band-pass filter from 4 - 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.00 Hz)
- Upper passband edge: 8.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 9.00 Hz)
- Filter length: 265 samples (1.656 s)

Setting up band-pass filter from 8 - 13 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth:

In [2]:
import pandas as pd
import numpy as np

# ZAKŁADAMY, ŻE OBIEKT `epochs` JEST JUŻ GOTOWY (po filtracji, ICA i odrzuceniu artefaktów)

print("Rozpoczynam ekstrakcję cech do pliku CSV...")

# ==========================================
# 1. EKSTRAKCJA CECH (Power Spectral Density)
# ==========================================
# Obliczamy gęstość widmową mocy (PSD) w kluczowym paśmie dla ruchu (8-30 Hz)
psd = epochs.compute_psd(method='welch', fmin=8.0, fmax=30.0, n_fft=256)

# Pobieramy dane jako macierz numpy. Kształt to: (liczba_epok, liczba_kanałów, liczba_częstotliwości)
psd_data = psd.get_data()

# Uśredniamy moc po wymiarze częstotliwości (oś 2).
# Otrzymujemy jedną wartość (średnią moc pasmową) dla każdego kanału w każdej epoce.
# Nowy kształt: (liczba_epok, liczba_kanałów) - idealny do tabeli!
features = np.mean(psd_data, axis=2)

# Logarytmizujemy wartości (standard w EEG, aby znormalizować rozkład danych)
features = np.log10(features)

# ==========================================
# 2. TWORZENIE TABELI (DataFrame)
# ==========================================
# Nazwy kolumn to nazwy elektrod (np. 'Cz', 'C3', 'C4' itd.)
df_features = pd.DataFrame(features, columns=epochs.ch_names)

# ==========================================
# 3. DODAWANIE ETYKIET (Mapowanie na "Task1", "Task2"...)
# ==========================================
# Pobieramy kody zdarzeń (ID) dla każdej epoki z MNE
event_ids = epochs.events[:, 2]

# W zbiorze PhysioNet występują domyślnie eventy T0, T1, T2.
# Twój kod UMAP oczekuje klas o nazwie "Task1", "Task2", itd.
# Tworzymy mapowanie (dostosuj je w zależności od tego, ile zadań analizujesz):
mne_event_mapping = epochs.event_id  # np. {'T0': 1, 'T1': 2, 'T2': 3}

# Tworzymy odwrotny słownik: ID_z_MNE -> Nowa nazwa dla Twojego CSV
# W PhysioNet: T0 to odpoczynek, T1 to np. lewa pięść, T2 to prawa pięść
label_map = {
    mne_event_mapping.get('T0', 1): "Task1",  # Odpoczynek
    mne_event_mapping.get('T1', 2): "Task2",  # Lewa ręka
    mne_event_mapping.get('T2', 3): "Task3"   # Prawa ręka
    # Jeśli załadujesz więcej runów, będą tu też T3, T4 itd. i przypiszesz je do Task4, Task5...
}

# Mapujemy ID na nowe nazwy "TaskX", a jeśli czegoś nie ma w słowniku, tworzymy etykietę dynamicznie
df_features['Label'] = [label_map.get(e, f"Task{e}") for e in event_ids]

# ==========================================
# 4. ZAPIS DO CSV
# ==========================================
file_name = 'dataset_eeg_features.csv'
df_features.to_csv(file_name, index=False)

print(f"Sukces! Wyekstrahowano {df_features.shape[1]-1} cech dla {df_features.shape[0]} próbek.")
print(f"Zapisano plik: {file_name}. Możesz teraz uruchomić swój kod UMAP.")

Rozpoczynam ekstrakcję cech do pliku CSV...
Effective window size : 1.600 (s)
Sukces! Wyekstrahowano 64 cech dla 87 próbek.
Zapisano plik: dataset_eeg_features.csv. Możesz teraz uruchomić swój kod UMAP.
